# Claim storage and comparison
Extract reference claims, save them for review, and compare new claims with manually verified records. Results are pandas DataFrames.

In [ ]:
from utils import setup_notebook, to_dataframe, write_json

from IPython.display import display

from data_platform.digest.claimify import extract_claims
from data_platform.digest.claim_store import ClaimStore

project_root = setup_notebook()


## 1. Configure

In [ ]:
run_live = False
store_path = project_root / ".data/claims-demo"
decision_model = "typesafe:jev-latest"
generation_model = "openai:gpt-6-luna"
embedding_model = "openai:text-embedding-3-small"
max_concurrency = 20
top_k = 5
store = ClaimStore(
    store_path,
    embedding_model=embedding_model,
    comparison_model=decision_model,
    max_concurrency=max_concurrency,
)
reference_markdown = (
    "# Acme annual report\n\n## Results\n\nAcme's revenue for 2025 was USD 10 million."
)
incoming_markdown = (
    "# Acme follow-up\n\n## Results\n\nAcme's revenue for 2025 was USD 12 million."
)


## 2. Extract and save reference claims
The saved records are unverified. Inspect their original sentences before approval.

In [ ]:
reference_records = []
if run_live:
    reference_result = await extract_claims(
        reference_markdown,
        decision_model=decision_model,
        generation_model=generation_model,
        max_concurrency=max_concurrency,
    )
    reference_records = store.store_unverified(
        reference_result, source="demo:acme-annual-report"
    )

reference_df = to_dataframe(
    reference_records,
    columns=["id", "text", "original_sentence", "source", "start", "end"],
)
display(reference_df)


## 3. Review and approve manually
Pause here. Inspect JSON files in `.data/claims-demo/unverified/`, then move only approved files to `verified/`. Rerun this cell to refresh the table. Verification means your approval, not automated fact-checking.

In [ ]:
verified_df = to_dataframe(
    store.list_claims("verified"),
    columns=["id", "text", "original_sentence", "source"],
)
display(verified_df)


## 4. Retrieve related claims
Only verified records enter retrieval. Index synchronization is automatic.

In [ ]:
query = "Acme's revenue for 2025 was USD 12 million."
matches = await store.retrieve(query, top_k=top_k) if run_live else []
matches_df = to_dataframe(
    [
        {
            "similarity": match.similarity,
            "text": match.claim.text,
            "source": match.claim.source,
            "id": match.claim.id,
        }
        for match in matches
    ],
    columns=["similarity", "text", "source", "id"],
)
display(matches_df)


## 5. Compare incoming claims
The incoming assertions are saved as unverified. An empty verified store yields no comparisons; verdicts do not approve either claim.

In [ ]:
report = None
if run_live:
    incoming_result = await extract_claims(
        incoming_markdown,
        decision_model=decision_model,
        generation_model=generation_model,
        max_concurrency=max_concurrency,
    )
    incoming_records = store.store_unverified(
        incoming_result, source="demo:acme-follow-up"
    )
    report = await store.check(incoming_records, top_k=top_k)

comparisons_df = to_dataframe(
    [
        {
            "verdict": pair.decision.verdict,
            "similarity": pair.similarity,
            "claim": pair.claim.text,
            "verified_claim": pair.verified_claim.text,
            "claim_source": pair.claim.source,
            "verified_source": pair.verified_claim.source,
        }
        for pair in (report.comparisons if report is not None else [])
    ],
    columns=[
        "verdict",
        "similarity",
        "claim",
        "verified_claim",
        "claim_source",
        "verified_source",
    ],
)
display(comparisons_df)
display(comparisons_df.loc[comparisons_df["verdict"].isin(["conflict", "uncertain"])])


## 6. Export (optional)
Saving replaces the comparison CSV and full report JSON. Absence of a conflict applies only to retrieved candidates.

In [ ]:
save_output = False
if save_output and report is not None:
    output_dir = project_root / ".data/reports"
    output_dir.mkdir(parents=True, exist_ok=True)
    comparisons_df.to_csv(output_dir / "comparisons.csv", index=False)
    write_json(output_dir / "comparisons.json", report)
